In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import random
import heapq
import math
import sys
from collections import defaultdict, deque, Counter
from itertools import combinations, permutations


class Problem(object):
    """The abstract class for a formal problem. A new domain subclasses this,
    overriding `actions` and `results`, and perhaps other methods.
    The default heuristic is 0 and the default action cost is 1 for all states.
    When yiou create an instance of a subclass, specify `initial`, and `goal` states 
    (or give an `is_goal` method) and perhaps other keyword args for the subclass."""

    def __init__(self, initial=None, goal=None, **kwds): 
        self.__dict__.update(initial=initial, goal=goal, **kwds) 
        
    def actions(self, state):        raise NotImplementedError
    def result(self, state, action): raise NotImplementedError
    def is_goal(self, state):        return state == self.goal
    def action_cost(self, s, a, s1): return 1
    def h(self, node):               return 0
    
    def __str__(self):
        return '{}({!r}, {!r})'.format(
            type(self).__name__, self.initial, self.goal)
    

class Node:
    "A Node in a search tree."
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.__dict__.update(state=state, parent=parent, action=action, path_cost=path_cost)

    def __repr__(self): return '<{}>'.format(self.state)
    def __len__(self): return 0 if self.parent is None else (1 + len(self.parent))
    def __lt__(self, other): return self.path_cost < other.path_cost
    
    
failure = Node('failure', path_cost=math.inf) # Indicates an algorithm couldn't find a solution.
cutoff  = Node('cutoff',  path_cost=math.inf) # Indicates iterative deepening search was cut off.
    
    
def expand(problem, node):
    "Expand a node, generating the children nodes."
    s = node.state
    for action in problem.actions(s):
        s1 = problem.result(s, action)
        cost = node.path_cost + problem.action_cost(s, action, s1)
        yield Node(s1, node, action, cost)
        

def path_states(node):
    "The sequence of states to get to this node."
    if node in (cutoff, failure, None): 
        return []
    return path_states(node.parent) + [node.state]


#Queues

FIFOQueue = deque

LIFOQueue = list

class PriorityQueue:
    """A queue in which the item with minimum f(item) is always popped first."""

    def __init__(self, items=(), key=lambda x: x): 
        self.key = key
        self.items = [] # a heap of (score, item) pairs
        for item in items:
            self.add(item)
         
    def add(self, item):
        """Add item to the queuez."""
        pair = (self.key(item), item)
        heapq.heappush(self.items, pair)

    def pop(self):
        """Pop and return the item with min f(item) value."""
        return heapq.heappop(self.items)[1]
    
    def top(self): return self.items[0][1]

    def __len__(self): return len(self.items)

In [2]:
#Breadth first search explores level by level and finds the fastes solution 
def breadth_first_search(problem):
    "Search shallowest nodes in the search tree first."
    node = Node(problem.initial)
    if problem.is_goal(problem.initial):
        return node
    frontier = FIFOQueue([node])
    reached = {problem.initial}
    while frontier:
        node = frontier.pop()
        for child in expand(problem, node):
            s = child.state
            if problem.is_goal(s):
                return child
            if s not in reached:
                reached.add(s)
                frontier.appendleft(child)
    return failure


def hamming_distance(A, B):
    "Number of positions where vectors A and B are different."
    return sum(a != b for a, b in zip(A, B))
    
    
def board9(state):
    "A string representing an puzzle board"
    return '{} {} {}\n{} {} {}\n{} {} {}\n'.format(*state)

In [11]:
class MagicSquarePuzzle(Problem):
    def __init__(self, initial, goal = (4, 9, 2, 3, 5, 7, 8, 1, 6)):
        super().__init__(initial=initial, goal = goal)
        
    def actions(self, state):
        return [(i, j) for i in range(9) for j in range (i + 1, 9)]
    
    def result(self, state, action):
        i, j = action
        s = list(state)
        s[i], s[j] = s[j], s[i]
        return tuple(s)
    
    def h1(self, node):
        """The misplaced tiles heuristic."""
        return hamming_distance(node.state, self.goal)
    
    def h2(self, node):
        """The Manhattan heuristic."""
        X = (0, 1, 2, 0, 1, 2, 0, 1, 2)
        Y = (0, 0, 0, 1, 1, 1, 2, 2, 2)
        total = 0 
        for tile in range(1, 10):
            curr = node.state.index(tile)
            goal = self.goal.index(tile)
            total += abs(X[curr] - X[goal]) + abs(Y[curr] - Y[goal])
        return total
                  
    def h(self, node): 
        return self.h2(node) 
        

In [12]:
p1 = MagicSquarePuzzle(initial=(1, 2, 3, 4, 5, 6, 7, 8, 9))

solution = breadth_first_search(p1)
print("solution found!")
print(board9(solution.state))
print("moves needed", len(solution))

print("path from start to solution")
for i, s in enumerate(path_states(solution)):
    print(f"step {i}")
    print(board9(s))

solution found!
4 9 2
3 5 7
8 1 6

moves needed 7
path from start to solution
step 0
1 2 3
4 5 6
7 8 9

step 1
2 1 3
4 5 6
7 8 9

step 2
3 1 2
4 5 6
7 8 9

step 3
4 1 2
3 5 6
7 8 9

step 4
4 6 2
3 5 1
7 8 9

step 5
4 9 2
3 5 1
7 8 6

step 6
4 9 2
3 5 7
1 8 6

step 7
4 9 2
3 5 7
8 1 6



In [14]:
class MagicSquarePuzzleV2(MagicSquarePuzzle):
    def is_goal(self, state):
        return(
            state[0] + state[1] + state[2] == 15 and
            state[3] + state[4] + state[5] == 15 and
            state[6] + state[7] + state[8] == 15 and
            
            state[0] + state[3] + state[6] == 15 and
            state[1] + state[4] + state[7] == 15 and
            state[2] + state[5] + state[8] == 15 and
            
            state[0] + state[4] + state[8] == 15 and
            state[2] + state[4] + state[6] == 15 
        )
    
checker = MagicSquarePuzzleV2(initial = (1, 2, 3, 4, 5, 6, 7, 8, 9))

all_solutions = [p for p in permutations(range(1, 10)) if checker.is_goal(p)]

print("total solution found", len(all_solutions))

for i, sol in enumerate(all_solutions, 1):
    print(f"solutions {i} : {sol}")
    print(board9(sol))

total solution found 8
solutions 1 : (2, 7, 6, 9, 5, 1, 4, 3, 8)
2 7 6
9 5 1
4 3 8

solutions 2 : (2, 9, 4, 7, 5, 3, 6, 1, 8)
2 9 4
7 5 3
6 1 8

solutions 3 : (4, 3, 8, 9, 5, 1, 2, 7, 6)
4 3 8
9 5 1
2 7 6

solutions 4 : (4, 9, 2, 3, 5, 7, 8, 1, 6)
4 9 2
3 5 7
8 1 6

solutions 5 : (6, 1, 8, 7, 5, 3, 2, 9, 4)
6 1 8
7 5 3
2 9 4

solutions 6 : (6, 7, 2, 1, 5, 9, 8, 3, 4)
6 7 2
1 5 9
8 3 4

solutions 7 : (8, 1, 6, 3, 5, 7, 4, 9, 2)
8 1 6
3 5 7
4 9 2

solutions 8 : (8, 3, 4, 1, 5, 9, 6, 7, 2)
8 3 4
1 5 9
6 7 2

